# ======================================================
# Project Name : TripGenie
#
# Your AI Travel Concierge
#
# Tech Stack:
# - LangGraph
# - LangChain
# - Groq
# - Gradio
# - External APIs (Weather, Places, etc.)
#
# Features:
# - Personalized travel planning
# - Travel dossier generation
# - Smart recommendations
# - Real-world API integrations
# - Genie themed user experience
#
# ======================================================

# Destination
# Destination Familiarity
# Traveller Type
# Budget
# Duration
# Travel Style
# Travel Vibe
# Food Preference
# Transport Preference
# Additional Wishes
# Weather
# Recommended Places
# Restaurants
# Hidden Gems
# Estimated Cost
# Travel Tips
# Itinerary
# Messages

In [76]:
!pip install langchain
!pip install langgraph
!pip install langchain_groq
!pip install gradio

In [77]:
# ======================================================
# Imports
# ======================================================

from typing import TypedDict, Annotated, List

from langgraph.graph import StateGraph, END

from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate

import gradio as gr

In [78]:
# ======================================================
# Planner State
# Stores all user inputs and AI generated information.
# ======================================================

class PlannerState(TypedDict):

    # User Inputs
    messages: Annotated[
        List[HumanMessage | AIMessage],
        "Stores the conversation history."
    ]

    destination: str
    destination_familiarity: str
    traveller_type: str
    budget: str
    duration: str
    travel_style: List[str]
    travel_vibe: List[str]
    food_preference: str
    transport_preference: str
    additional_wishes: str

    # AI Generated Information
    weather: str
    recommended_places: List[str]
    restaurants: List[str]
    hidden_gems: List[str]
    estimated_cost: str
    travel_tips: List[str]
    itinerary: str

In [79]:
# ======================================================
# LLM Initialization
# Initializes the Groq LLM used by TripGenie.
# ======================================================

from langchain_groq import ChatGroq

llm = ChatGroq(
    temperature=0,
    groq_api_key="YOUR_GROQ_API_KEY",
    model_name="llama-3.3-70b-versatile"
)

In [117]:
# ======================================================
# Gradio UI Components
# Defines all user input components for TripGenie.
# ======================================================

# Destination
destination_input = gr.Textbox(
    label="Where shall we wander today?"
)

# Destination Familiarity
destination_familiarity_input = gr.Radio(
    choices=[
        "First Time Visitor",
        "Been Here Before",
        "Local Resident"
    ],
    label="How familiar are you with this destination?"
)

# Traveller Type
traveller_type_input = gr.Radio(
    choices=[
        "Solo",
        "Couple",
        "Friends",
        "Family"
    ],
    label="Who's travelling?"
)

# Budget
budget_input = gr.Radio(
    choices=[
        "Budget Friendly",
        "Moderate",
        "Premium",
        "Luxury"
    ],
    label="What's your budget?"
)

# Trip Duration
duration_input = gr.Radio(
    choices=[
        "Half Day",
        "Full Day",
        "Weekend",
        "2-3 Days",
        "1 Week"
    ],
    label="How long is your adventure?"
)

# Travel Style
travel_style_input = gr.CheckboxGroup(
    choices=[
        "Adventure",
        "Food",
        "Luxury",
        "Shopping",
        "Historical",
        "Nature",
        "Nightlife",
        "Photography",
        "Relaxation",
        "Spiritual",
        "Local Culture",
        "Hidden Gems"
    ],
    label="What experiences are you looking for?"
)

# Travel Vibe
travel_vibe_input = gr.CheckboxGroup(
    choices=[
        "Chill",
        "Romantic",
        "Party",
        "Fast Paced",
        "Luxury",
        "Offbeat",
        "Family Time",
        "Cultural"
    ],
    label="What's the vibe?"
)

# Food Preference
food_preference_input = gr.Dropdown(
    choices=[
        "Vegetarian",
        "Non-Vegetarian",
        "Vegan",
        "Jain",
        "No Preference"
    ],
    label="Food Preference"
)

# Transport Preference
transport_preference_input = gr.Dropdown(
    choices=[
        "Walking",
        "Metro",
        "Cab",
        "Self Drive",
        "Bike Rental",
        "Public Transport",
        "No Preference"
    ],
    label="Preferred Mode of Transport"
)

# Additional Wishes
additional_wishes_input = gr.Textbox(
    lines=5,
    label="Tell TripGenie your wishes (Optional)"
)

# Submit Button
plan_trip_button = gr.Button(
    "Plan My Adventure"
)

# ======================================================
# Prompt Templates
# Defines the prompts used by TripGenie.
# ======================================================

In [118]:
# ======================================================
# Main Travel Planner Prompt
# ======================================================

travel_planner_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are TripGenie, an AI Travel Concierge.

            Your job is to grant the user's travel wishes and create a premium Travel Dossier.

            Generate the output ONLY in the following format:

            1. YOUR WISH HAS BEEN GRANTED

            2. TRIP SUMMARY
               - Brief summary of the trip.

            3. MUST VISIT PLACES
              - Recommend a maximum of 5 places that best suit the user's preferences, travel style, and trip vibe.

            4. DAY-WISE ADVENTURE PLAN
               - Create an optimized day-wise travel plan.
               - Group nearby places together whenever possible.

            5. FOOD RECOMMENDATIONS
               - Recommend a maximum of 3 places to eat.

            6. HIDDEN GEMS
               - Recommend a maximum of 3 hidden gems.

            7. TRANSPORTATION GUIDE
               - Suggest the best mode of transportation.

            8. BUDGET INSIGHTS
               - Briefly explain how the user's budget suits the trip.

            9. PACKING SUGGESTIONS
               - Suggest useful items to carry.

            10. TRAVEL TIPS
                - Provide useful travel tips.

            11. TRIPGENIE'S FINAL WISH
                - Give one personalized recommendation to make the trip more memorable.

            Rules:
                - Do not overwhelm the user.
                - Prioritize quality over quantity.
                - Behave like a premium travel concierge.
                - Keep the output well structured.
                - If the user provides specific wishes, intelligently incorporate them.
                - Never recommend overly touristy places unless they genuinely match the user's preferences.
                - Prioritize hidden gems, lesser-known experiences, and locally loved places whenever possible.
                - Optimize the trip by grouping nearby places together.
            """
        ),

        (
            "human",
            """
            Destination: {destination}

            Destination Familiarity: {destination_familiarity}

            Traveller Type: {traveller_type}

            Budget: {budget}

            Duration: {duration}

            Travel Style: {travel_style}

            Travel Vibe: {travel_vibe}

            Food Preference: {food_preference}

            Transport Preference: {transport_preference}

            Additional Wishes:
            {additional_wishes}
            """
        )
    ]
)

In [119]:
# ======================================================
# Initialize Planner State
# Creates the initial state for every TripGenie request.
# ======================================================

def initialize_state() -> PlannerState:

    return {

        # User Inputs
        "messages": [],
        "destination": "",
        "destination_familiarity": "",
        "traveller_type": "",
        "budget": "",
        "duration": "",
        "travel_style": [],
        "travel_vibe": [],
        "food_preference": "",
        "transport_preference": "",
        "additional_wishes": "",

        # AI Generated Information
        "weather": "",
        "recommended_places": [],
        "restaurants": [],
        "hidden_gems": [],
        "estimated_cost": "",
        "travel_tips": [],

        # Travel Dossier Components
        "trip_summary": "",
        "must_visit_places": [],
        "day_wise_itinerary": "",
        "food_recommendations": [],
        "transport_guide": "",
        "packing_suggestions": [],
        "final_wish": ""

    }

In [120]:
# ======================================================
# Collect User Preferences Node
# Stores all user inputs inside PlannerState.
# ======================================================

def collect_user_preferences(

    destination,
    destination_familiarity,
    traveller_type,
    budget,
    duration,
    travel_style,
    travel_vibe,
    food_preference,
    transport_preference,
    additional_wishes

) -> PlannerState:


    state = initialize_state()


    state["destination"] = destination
    state["destination_familiarity"] = destination_familiarity
    state["traveller_type"] = traveller_type
    state["budget"] = budget
    state["duration"] = duration
    state["travel_style"] = travel_style
    state["travel_vibe"] = travel_vibe
    state["food_preference"] = food_preference
    state["transport_preference"] = transport_preference
    state["additional_wishes"] = additional_wishes


    return state

In [121]:
# ======================================================
# Travel Planner Node
# Creates a personalized travel plan using the LLM.
# ======================================================

def travel_planner_node(state: PlannerState) -> PlannerState:

    response = llm.invoke(

        travel_planner_prompt.format_messages(

            destination=state["destination"],
            destination_familiarity=state["destination_familiarity"],
            traveller_type=state["traveller_type"],
            budget=state["budget"],
            duration=state["duration"],
            travel_style=", ".join(state["travel_style"]),
            travel_vibe=", ".join(state["travel_vibe"]),
            food_preference=state["food_preference"],
            transport_preference=state["transport_preference"],
            additional_wishes=state["additional_wishes"]

        )

    )

    # Temporarily store the complete Travel Dossier
    state["trip_summary"] = response.content

    # Store the AI response in conversation history
    state["messages"].append(
        AIMessage(content=response.content)
    )

    return state

In [122]:
# ======================================================
# Testing TripGenie
# ======================================================

# Create an empty state
state = initialize_state()

# Add dummy user inputs
state["destination"] = "Delhi"
state["destination_familiarity"] = "First Time Visitor"
state["traveller_type"] = "Friends"
state["budget"] = "Moderate"
state["duration"] = "2-3 Days"
state["travel_style"] = ["Food", "Photography", "Historical"]
state["travel_vibe"] = ["Chill", "Cultural"]
state["food_preference"] = "Vegetarian"
state["transport_preference"] = "Metro"
state["additional_wishes"] = (
    "I really want to explore hidden gems and enjoy good street food. "
    "I don't mind walking a lot and would love some photography spots."
)

# Generate the travel plan
state = travel_planner_node(state)

# Print the generated Travel Dossier
print(state["trip_summary"])

1. YOUR WISH HAS BEEN GRANTED

2. TRIP SUMMARY
   - Explore the vibrant city of Delhi with your friends, discovering its rich history, cultural heritage, and delicious vegetarian street food. This 2-3 day trip is perfect for a chill and cultural experience, with plenty of opportunities for photography and walking.

3. MUST VISIT PLACES
   - Red Fort: A historic fort with stunning architecture and photography opportunities.
   - Hauz Khas Village: A charming neighborhood with a mix of history, street food, and trendy cafes.
   - Qutub Minar: A UNESCO World Heritage Site and one of the tallest minarets in India.
   - Chandni Chowk: A bustling street market with a variety of street food and shopping options.
   - Lodhi Gardens: A beautiful park with historic tombs and plenty of green space for relaxation.

4. DAY-WISE ADVENTURE PLAN
   - Day 1: Start at Red Fort, then walk to Chandni Chowk for lunch and exploration. End the day with a visit to Qutub Minar for sunset photography.
   - Day 

# ======================================================
# TripGenie Backend Function
# Connects the UI with the Travel Planner.
# ======================================================

In [123]:
def generate_trip(

    destination,
    destination_familiarity,
    traveller_type,
    budget,
    duration,
    travel_style,
    travel_vibe,
    food_preference,
    transport_preference,
    additional_wishes

):

    # Collect all user preferences
    state = collect_user_preferences(

        destination,
        destination_familiarity,
        traveller_type,
        budget,
        duration,
        travel_style,
        travel_vibe,
        food_preference,
        transport_preference,
        additional_wishes

    )

    # Generate the Travel Dossier
    state = travel_planner_node(state)

    # Generate Destination Intelligence
    destination_guide = generate_destination_intelligence(
        destination
    )

    # Split the Travel Dossier into sections
    sections = parse_travel_dossier(
        state["trip_summary"]
    )

    # Extract recommended places

    recommended_places = extract_recommended_places(
        sections["must_visit_places"]
    )

    # Get only the place names

    place_names = [

        place["name"]

        for place in recommended_places

    ]

    # Generate Places Intelligence

    places_guide = generate_places_intelligence(
        place_names
    )

    # Merge sections for the Tab UI

    overview = sections["trip_summary"]

    places = places_guide

    itinerary = sections["day_plan"]

    food = sections["food"]

    budget = (
        sections["transport"]
        + "\n\n"
        + sections["budget"]
    )

    hidden_gems = sections["hidden_gems"]

    travel_tips = (
        sections["packing"]
        + "\n\n"
        + sections["travel_tips"]
    )

    final_wish = sections["final_wish"]

    # Return all tabs to Gradio

    return (

        overview,
        destination_guide,
        places,
        itinerary,
        food,
        budget,
        hidden_gems,
        travel_tips,
        final_wish

    )

# ======================================================
# Travel Intelligence Engine
# Enriches TripGenie's recommendations with additional information.
# ======================================================

In [124]:
def extract_recommended_places(must_visit_places):

    places = []

    for line in must_visit_places.split("\n"):

        line = line.strip()

        if line.startswith("-"):

            place_name = line.replace("-", "").strip()

            places.append(
                {
                    "name": place_name
                }
            )

    return places

# ======================================================
# Travel Intelligence Engine
# Enriches TripGenie's recommendations with additional information.
# ======================================================

# ======================================================
# Destination Intelligence Module
# Generates useful travel information for the destination.
# ======================================================

In [125]:
def generate_destination_intelligence(destination):

    response = llm.invoke(

        f"""

        You are a world-class travel guide.

        Provide the following information about {destination}.

        1. Best Time to Visit
        2. Safety Tips
        3. Local Etiquette
        4. Packing Suggestions

        Keep the response:
        - Short and practical.
        - Tourist friendly.
        - Well formatted.
        - Maximum 2-3 bullet points per section.

        """

    )

    return response.content

# ======================================================
# Destination Intelligence Module
# ======================================================

In [126]:
def generate_place_intelligence(place_name):

    response = llm.invoke(

        f"""

        You are a world-class travel guide.

        Provide the following information about {place_name}.

        1. Why should a tourist visit this place?
        2. Best time to visit.
        3. Approximate time required to explore.
        4. One interesting or fun fact about the place.

        Keep the response:
        - Tourist friendly.
        - Short and practical.
        - Well formatted.
        - Maximum 2-3 lines per point.

        """

    )

    return response.content


def generate_places_intelligence(places):

    response = llm.invoke(

        f"""

        You are a world-class travel guide.

        Provide the following information for each of these tourist attractions:

        {', '.join(places)}

        For every place, provide:

        1. Why should a tourist visit?
        2. Best time to visit.
        3. Approximate time required to explore.
        4. One interesting or fun fact.

        Keep the response:
        - Tourist friendly.
        - Short and practical.
        - Well formatted.
        - Maximum 2-3 lines per point.

        """

    )

    return response.content

# ======================================================
# Travel Dossier Parser
# Splits TripGenie's response into individual sections.
# ======================================================

In [127]:
def parse_travel_dossier(response):

    sections = {

        "trip_summary": "",
        "must_visit_places": "",
        "day_plan": "",
        "food": "",
        "hidden_gems": "",
        "transport": "",
        "budget": "",
        "packing": "",
        "travel_tips": "",
        "final_wish": ""

    }

    # Split the response using our headings

    if "2. TRIP SUMMARY" in response:
        sections["trip_summary"] = response.split(
            "2. TRIP SUMMARY"
        )[1].split(
            "3. MUST VISIT PLACES"
        )[0].strip()


    if "3. MUST VISIT PLACES" in response:
        sections["must_visit_places"] = response.split(
            "3. MUST VISIT PLACES"
        )[1].split(
            "4. DAY-WISE ADVENTURE PLAN"
        )[0].strip()


    if "4. DAY-WISE ADVENTURE PLAN" in response:
        sections["day_plan"] = response.split(
            "4. DAY-WISE ADVENTURE PLAN"
        )[1].split(
            "5. FOOD RECOMMENDATIONS"
        )[0].strip()


    if "5. FOOD RECOMMENDATIONS" in response:
        sections["food"] = response.split(
            "5. FOOD RECOMMENDATIONS"
        )[1].split(
            "6. HIDDEN GEMS"
        )[0].strip()


    if "6. HIDDEN GEMS" in response:
        sections["hidden_gems"] = response.split(
            "6. HIDDEN GEMS"
        )[1].split(
            "7. TRANSPORTATION GUIDE"
        )[0].strip()


    if "7. TRANSPORTATION GUIDE" in response:
        sections["transport"] = response.split(
            "7. TRANSPORTATION GUIDE"
        )[1].split(
            "8. BUDGET INSIGHTS"
        )[0].strip()


    if "8. BUDGET INSIGHTS" in response:
        sections["budget"] = response.split(
            "8. BUDGET INSIGHTS"
        )[1].split(
            "9. PACKING SUGGESTIONS"
        )[0].strip()


    if "9. PACKING SUGGESTIONS" in response:
        sections["packing"] = response.split(
            "9. PACKING SUGGESTIONS"
        )[1].split(
            "10. TRAVEL TIPS"
        )[0].strip()


    if "10. TRAVEL TIPS" in response:
        sections["travel_tips"] = response.split(
            "10. TRAVEL TIPS"
        )[1].split(
            "11. TRIPGENIE'S FINAL WISH"
        )[0].strip()


    if "11. TRIPGENIE'S FINAL WISH" in response:
        sections["final_wish"] = response.split(
            "11. TRIPGENIE'S FINAL WISH"
        )[1].strip()


    return sections

In [128]:
# ======================================================
# Trip Profile Builder
# Creates a live summary of the user's travel preferences.
# ======================================================

def build_review(

    destination,
    destination_familiarity,
    traveller_type,
    budget,
    duration,
    travel_style,
    travel_vibe,
    food_preference,
    transport_preference,
    additional_wishes

):

    travel_style_text = (
        ", ".join(travel_style)
        if travel_style
        else "Not Selected"
    )

    travel_vibe_text = (
        ", ".join(travel_vibe)
        if travel_vibe
        else "Not Selected"
    )

    return f"""
### Trip Summary

- **Destination:** {destination or "Not Selected"}
- **Destination Familiarity:** {destination_familiarity or "Not Selected"}
- **Traveller Type:** {traveller_type or "Not Selected"}
- **Budget:** {budget or "Not Selected"}
- **Duration:** {duration or "Not Selected"}
- **Travel Experiences:** {travel_style_text}
- **Travel Mood:** {travel_vibe_text}
- **Food Preference:** {food_preference or "Not Selected"}
- **Transport Preference:** {transport_preference or "Not Selected"}

### Additional Wishes

{additional_wishes or "No additional wishes provided."}
"""

# ======================================================
# TripGenie User Interface
# Creates the complete Gradio UI.
# ======================================================

In [129]:
with gr.Blocks(
    title="TripGenie"
) as demo:

    # ======================================================
    # Header
    # ======================================================

    gr.Markdown("# TripGenie")

    gr.Markdown(
        """
### Your Personal AI Travel Concierge

Build personalized travel experiences in seconds.
"""
    )

    # ======================================================
    # Destination
    # ======================================================

    destination = gr.Textbox(
        label="Destination",
        placeholder="Delhi, Tokyo, Paris..."
    )

    # ======================================================
    # Destination Familiarity + Traveller Type
    # ======================================================

    with gr.Row():

        destination_familiarity = gr.Radio(
            choices=[
                "First Time Visitor",
                "Been Here Before",
                "Local Resident"
            ],
            label="Destination Familiarity"
        )

        traveller_type = gr.Radio(
            choices=[
                "Solo",
                "Couple",
                "Friends",
                "Family"
            ],
            label="Traveller Type"
        )

    # ======================================================
    # Budget + Duration
    # ======================================================

    with gr.Row():

        budget = gr.Radio(
            choices=[
                "Budget Friendly",
                "Moderate",
                "Premium",
                "Luxury"
            ],
            label="Budget"
        )

        duration = gr.Radio(
            choices=[
                "Half Day",
                "Full Day",
                "Weekend",
                "2-3 Days",
                "1 Week"
            ],
            label="Duration"
        )

    # ======================================================
    # Travel Experiences
    # ======================================================

    travel_style = gr.CheckboxGroup(
        choices=[
            "Adventure",
            "Food",
            "Luxury",
            "Shopping",
            "Historical",
            "Nature",
            "Nightlife",
            "Photography",
            "Relaxation",
            "Spiritual",
            "Local Culture",
            "Hidden Gems"
        ],
        label="Travel Experiences"
    )

    # ======================================================
    # Travel Mood
    # ======================================================

    travel_vibe = gr.CheckboxGroup(
        choices=[
            "Chill",
            "Romantic",
            "Party",
            "Fast Paced",
            "Luxury",
            "Offbeat",
            "Family Time",
            "Cultural"
        ],
        label="Travel Mood"
    )

    # ======================================================
    # Food Preference + Transport Preference
    # ======================================================

    with gr.Row():

        food_preference = gr.Dropdown(
            choices=[
                "Vegetarian",
                "Non-Vegetarian",
                "Vegan",
                "Jain",
                "No Preference"
            ],
            label="Food Preference"
        )

        transport_preference = gr.Dropdown(
            choices=[
                "Walking",
                "Metro",
                "Cab",
                "Self Drive",
                "Bike Rental",
                "Public Transport",
                "No Preference"
            ],
            label="Transport Preference"
        )

    # ======================================================
    # Additional Wishes
    # ======================================================

    additional_wishes = gr.Textbox(
        lines=3,
        label="Additional Wishes (Optional)",
        placeholder="Tell TripGenie anything you'd like it to know about your ideal trip..."
    )

    # ======================================================
    # Plan Trip Button
    # ======================================================

    plan_trip_button = gr.Button(
        "Plan My Trip"
    )

    # ======================================================
    # Travel Dossier
    # ======================================================

    gr.Markdown("---")

    gr.Markdown(
        "## Your Travel Dossier"
    )

    # ======================================================
    # Travel Dossier Tabs
    # ======================================================

    with gr.Tabs():

        with gr.Tab("Overview"):
            overview_output = gr.Markdown()

        with gr.Tab("Destination Guide"):
            destination_guide_output = gr.Markdown()

        with gr.Tab("Places"):
            places_output = gr.Markdown()

        with gr.Tab("Itinerary"):
            itinerary_output = gr.Markdown()

        with gr.Tab("Food"):
            food_output = gr.Markdown()

        with gr.Tab("Budget"):
            budget_output = gr.Markdown()

        with gr.Tab("Hidden Gems"):
            hidden_gems_output = gr.Markdown()

        with gr.Tab("Travel Tips"):
            travel_tips_output = gr.Markdown()

        with gr.Tab("Final Wish"):
            final_wish_output = gr.Markdown()

    # ======================================================
    # Connect Button with Backend
    # ======================================================

    plan_trip_button.click(

        fn=generate_trip,

        inputs=[

            destination,
            destination_familiarity,
            traveller_type,
            budget,
            duration,
            travel_style,
            travel_vibe,
            food_preference,
            transport_preference,
            additional_wishes

        ],

        outputs=[

            overview_output,
            destination_guide_output,
            places_output,
            itinerary_output,
            food_output,
            budget_output,
            hidden_gems_output,
            travel_tips_output,
            final_wish_output

        ]

    )

In [130]:
# ======================================================
# Launch TripGenie
# Runs the Gradio application.
# ======================================================

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fd98e56eaec88dd21c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
